# 01b – Unpivot dữ liệu lâm sàng (tùy chọn)

**Mục đích:** chuyển file điều trị lâm sàng do giảng viên cung cấp (oxy, kháng sinh tại nhập viện / 24 giờ / 48 giờ, số giờ thở máy) từ dạng rộng sang dạng dài (mục 3.4).

**Trạng thái:** *chưa có file thật.* Nhóm mới chỉ có ví dụ mô tả. Notebook này được viết theo mô tả đó và **đã kiểm thử trên dữ liệu giả dựng tại chỗ (ô 2b)**, không phải trên dữ liệu của giảng viên. Khi có file thật:
1. Đặt file vào `data/raw/` và sửa `CLINICAL_FILE`, `HEADER_ROWS` ở ô cấu hình.
2. Chạy lại, đọc phần **cột không nhận diện được** và **tên thuốc chưa khớp** rồi sửa các biểu thức/từ điển ở ô cấu hình.
3. Hỏi giảng viên một ô trống nghĩa là gì (chưa ghi nhận hay không dùng thuốc) rồi chỉnh `NO_DRUG_TOKENS`, `MISSING_TOKENS`.

**Đầu ra** (`data/anonymized/`): `clinical_oxygen_long.csv`, `clinical_antibiotic_long.csv`, `clinical_ventilation.csv`.

In [ ]:
from pathlib import Path
import os, re, hmac, hashlib, difflib
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
ANON = ROOT / 'data' / 'anonymized'
PROC = ROOT / 'data' / 'processed'

# ---- Cấu hình: sửa theo file thật ----
CLINICAL_FILE = RAW / 'clinical_treatment.xlsx'     # tên file giả định
HEADER_ROWS = [0, 1]                                # số dòng tiêu đề (2 = tiêu đề hai tầng)

PATIENT_COL_RE = r'patient|id|ma_?bn|mã'
VENT_COL_RE = r'vent|thở máy|tho may'
TIMEPOINTS = {                                       # biểu thức nhận diện → mốc chuẩn
    'admission': r'admi|nhập viện|nhap vien|\b0\s*h',
    '24h': r'24',
    '48h': r'48',
}
NO_DRUG_TOKENS = {'none', 'no', 'nil', 'không', 'khong', 'no antibiotic', 'không dùng'}
MISSING_TOKENS = {'', 'nan', 'na', 'n/a', 'null', 'unknown', '?', 'chưa rõ', 'chua ro'}

# Từ điển tên thuốc chuẩn (mở rộng khi gặp tên mới) và bí danh/lỗi chính tả đã biết
CANONICAL_DRUGS = ['Ertapenem', 'Meropenem', 'Imipenem', 'Linezolid', 'Metronidazole', 'Vancomycin',
                   'Ceftriaxone', 'Cefepime', 'Piperacillin-Tazobactam', 'Azithromycin', 'Levofloxacin',
                   'Ciprofloxacin', 'Colistin', 'Amoxicillin-Clavulanate', 'Doxycycline']
DRUG_ALIASES = {'etarpenem': 'Ertapenem', 'ertapenem': 'Ertapenem', 'linezolide': 'Linezolid',
                'metronidazol': 'Metronidazole', 'piperacillin/tazobactam': 'Piperacillin-Tazobactam',
                'pip-tazo': 'Piperacillin-Tazobactam', 'augmentin': 'Amoxicillin-Clavulanate'}
FUZZY_CUTOFF = 0.85
OXYGEN_ALIASES = {'ra': 'Room air', 'room air': 'Room air', 'nasal cannula': 'Nasal cannula', 'nc': 'Nasal cannula',
                  'mask': 'Face mask', 'face mask': 'Face mask', 'hfnc': 'High-flow nasal cannula',
                  'niv': 'Non-invasive ventilation', 'bipap': 'Non-invasive ventilation',
                  'intubated': 'Invasive ventilation', 'mechanical ventilation': 'Invasive ventilation'}

# Muối băm dùng chung với notebook 02 (nếu file có mã bệnh nhân định danh được)
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
SALT = os.getenv('ANON_SALT')

def pseudonymize(value):
    return hmac.new(SALT.encode(), str(value).encode(), hashlib.sha256).hexdigest()[:16]

## 1. Hàm xử lý

In [ ]:
def flatten_columns(df):
    """Làm phẳng tiêu đề nhiều tầng: điền tiếp tầng trên cho ô 'Unnamed', nối các tầng bằng ' | '."""
    if not isinstance(df.columns, pd.MultiIndex):
        df.columns = [str(c).strip() for c in df.columns]
        return df
    levels = pd.DataFrame(df.columns.tolist())
    levels = levels.apply(lambda col: col.map(lambda x: np.nan if str(x).startswith('Unnamed') else str(x).strip()))
    levels[0] = levels[0].ffill()                      # tầng trên bị gộp ô → điền tiếp
    df.columns = [' | '.join(dict.fromkeys(str(x) for x in row if pd.notna(x))) for row in levels.values]
    return df

def classify_value(raw):
    """Phân biệt: 'missing' (không ghi nhận) – 'none' (ghi rõ không dùng) – 'given' (có dùng)."""
    if pd.isna(raw): return 'missing'
    t = str(raw).strip().lower()
    if t in MISSING_TOKENS: return 'missing'
    if t in NO_DRUG_TOKENS: return 'none'
    return 'given'

def normalize_drug(name):
    """Trả về (tên chuẩn, cách khớp). Không đoán bừa: dưới ngưỡng thì giữ nguyên và đánh dấu 'unmatched'."""
    key = re.sub(r'\s+', ' ', name.strip().lower())
    if key in DRUG_ALIASES:
        return DRUG_ALIASES[key], 'alias'
    for c in CANONICAL_DRUGS:
        if key == c.lower():
            return c, 'exact'
    hit = difflib.get_close_matches(key, [c.lower() for c in CANONICAL_DRUGS], n=1, cutoff=FUZZY_CUTOFF)
    if hit:
        return next(c for c in CANONICAL_DRUGS if c.lower() == hit[0]), 'fuzzy'
    return name.strip().title(), 'unmatched'

def split_regimen(raw):
    """Tách phác đồ theo dấu '+': 'Linezolide + Metronidazol' → ['Linezolide', 'Metronidazol']."""
    return [p.strip() for p in str(raw).split('+') if p.strip()]

def timepoint_of(col):
    for tp, rx in TIMEPOINTS.items():
        if re.search(rx, col.lower()):
            return tp
    return None

def find_columns(df, topic_rx):
    """Các cột thuộc một chủ đề (oxy / kháng sinh) kèm mốc thời gian của chúng."""
    found, unknown = {}, []
    for c in df.columns:
        if re.search(topic_rx, c.lower()):
            tp = timepoint_of(c)
            (found.__setitem__(tp, c) if tp else unknown.append(c))
    return found, unknown

## 2. Quy trình chính

In [ ]:
def process(df):
    df = flatten_columns(df.copy())
    pid_col = next(c for c in df.columns if re.search(PATIENT_COL_RE, c.lower()))
    oxy_cols, oxy_unknown = find_columns(df, r'oxy|oxi|thở oxy')
    abx_cols, abx_unknown = find_columns(df, r'antibio|kháng sinh|khang sinh|abx')
    vent_col = next((c for c in df.columns if re.search(VENT_COL_RE, c.lower())), None)

    pid = df[pid_col].astype(str).map(pseudonymize)
    unmapped_oxy = set()

    # Oxy: mỗi bệnh nhân một dòng cho mỗi mốc
    oxy_rows = []
    for tp, col in oxy_cols.items():
        for p, v in zip(pid, df[col]):
            status = classify_value(v)
            label = None
            if status == 'given':
                key = str(v).strip().lower()
                label = OXYGEN_ALIASES.get(key)
                if label is None:
                    unmapped_oxy.add(str(v).strip()); label = str(v).strip()
            oxy_rows.append((p, tp, label, status))
    oxygen = pd.DataFrame(oxy_rows, columns=['patient_hash', 'timepoint', 'oxygen_support', 'status'])

    # Kháng sinh: một dòng cho mỗi (bệnh nhân, mốc, thuốc)
    abx_rows = []
    for tp, col in abx_cols.items():
        for p, v in zip(pid, df[col]):
            status = classify_value(v)
            if status != 'given':
                abx_rows.append((p, tp, None, None, status)); continue
            for d in split_regimen(v):
                std, how = normalize_drug(d)
                abx_rows.append((p, tp, std, how, 'given'))
    antibiotic = pd.DataFrame(abx_rows, columns=['patient_hash', 'timepoint', 'drug_name', 'match_method', 'status'])

    ventilation = None
    if vent_col:
        ventilation = pd.DataFrame({'patient_hash': pid, 'ventilation_hours': pd.to_numeric(df[vent_col], errors='coerce')})
    report = {'cột không nhận diện được mốc': oxy_unknown + abx_unknown, 'tên oxy chưa có trong từ điển': sorted(unmapped_oxy),
              'tên thuốc chưa khớp': sorted(antibiotic.loc[antibiotic.match_method == 'unmatched', 'drug_name'].unique())}
    return oxygen, antibiotic, ventilation, report

### 2b. Kiểm thử trên dữ liệu giả (chỉ chạy khi chưa có file thật)
Dữ liệu dưới đây **tự dựng để kiểm thử logic**, không phải dữ liệu của giảng viên, và không được ghi ra đĩa.

In [ ]:
if not CLINICAL_FILE.exists():
    assert SALT, 'Cần ANON_SALT (xem notebook 02) để băm mã bệnh nhân.'
    cols = pd.MultiIndex.from_tuples([
        ('Patient', 'ID'),
        ('Oxygen support', 'Admission'), ('Oxygen support', '24h'), ('Oxygen support', '48h'),
        ('Antibiotic', 'Admission'), ('Antibiotic', '24h'), ('Antibiotic', '48h'),
        ('Ventilation', 'Hours')])
    demo = pd.DataFrame([
        ['A001', 'Nasal cannula', 'HFNC', 'NIV', 'Etarpenem', 'Etarpenem + Linezolide', 'Linezolide + Metronidazol', 12],
        ['A002', 'Room air', 'Room air', 'Room air', 'None', 'None', np.nan, 0],
        ['A003', 'Mask', np.nan, 'Intubated', 'Meropenem', np.nan, 'Vancomycin + Metronidazol', 96],
        ['A004', 'Oxygen tank xyz', 'RA', 'RA', 'Zzzmycin', 'None', 'None', 0]], columns=cols)
    oxygen, antibiotic, ventilation, report = process(demo)
    print(antibiotic.to_string()); print(report)

    # Kiểm tra logic chính
    a1 = antibiotic[(antibiotic.patient_hash == antibiotic.patient_hash.iloc[0]) & (antibiotic.timepoint == '24h')]
    assert sorted(a1.drug_name) == ['Ertapenem', 'Linezolid']                      # tách '+' và sửa chính tả
    assert (antibiotic.status == 'none').sum() == 4 and (antibiotic.status == 'missing').sum() == 2   # phân biệt không dùng và thiếu
    assert len(oxygen) == 4 * 3                                                    # mỗi bệnh nhân một dòng mỗi mốc
    assert report['tên thuốc chưa khớp'] == ['Zzzmycin'] and report['tên oxy chưa có trong từ điển'] == ['Oxygen tank xyz']
    print('Kiểm thử logic: đạt.')

## 3. Chạy trên file thật
Chỉ chạy khi `CLINICAL_FILE` tồn tại. **Không dùng kết quả cho báo cáo nếu danh sách `report` bên dưới còn mục chưa xử lý.**

In [ ]:
if CLINICAL_FILE.exists():
    assert SALT, 'Thiếu ANON_SALT'
    reader = pd.read_excel if CLINICAL_FILE.suffix.lower() in ('.xlsx', '.xls') else pd.read_csv
    raw_df = reader(CLINICAL_FILE, header=HEADER_ROWS, keep_default_na=False, na_values=[''])
    oxygen, antibiotic, ventilation, report = process(raw_df)
    print(report)
    ANON.mkdir(parents=True, exist_ok=True)
    oxygen.to_csv(ANON / 'clinical_oxygen_long.csv', index=False)
    antibiotic.to_csv(ANON / 'clinical_antibiotic_long.csv', index=False)
    if ventilation is not None:
        ventilation.to_csv(ANON / 'clinical_ventilation.csv', index=False)
    print(antibiotic.status.value_counts())
else:
    print('Chưa có file lâm sàng thật: bỏ qua bước ghi file.')